In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
import hashlib

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 3, Finished, Available, Finished, False)

In [2]:
df = spark.sql("SELECT * FROM MSGAnalytics.dbo.proposal_slide_content_raw")
display(df)

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 4, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 193a9b70-720b-4873-a2d3-6440537de4c1)

## Create a clean slide-level table

In [3]:
def clean_text(text):
    if text is None:
        return None

    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = text.strip()

    return text

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 5, Finished, Available, Finished, True)

In [4]:
clean_text_udf = udf(clean_text, StringType())

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 6, Finished, Available, Finished, True)

In [5]:
cleaned_df = (
    spark.table("proposal_slide_content_raw")
    .withColumn("cleaned_text", clean_text_udf(col("combined_text")))
    .withColumn("word_count", size(split(col("cleaned_text"), " ")))
)

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 7, Finished, Available, Finished, True)

In [6]:
display(cleaned_df)

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 8, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 6f71a091-d733-426d-9d56-60ca4b39443c)

### Add slide title to clean data

In [7]:
def detect_slide_title(text):
    if text is None:
        return None

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    if len(lines) == 0:
        return None

    return lines[0][:250]

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 9, Finished, Available, Finished, True)

In [8]:
detect_slide_title_udf = udf(detect_slide_title, StringType())

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 10, Finished, Available, Finished, True)

In [9]:
cleaned_df = cleaned_df.withColumn(
    "slide_title",
    detect_slide_title_udf(col("combined_text"))
)

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 11, Finished, Available, Finished, True)

In [10]:
display(cleaned_df)

StatementMeta(, 9159339a-1f13-4a37-81a5-417f9e0f8be9, 12, Submitted, Running, Running, True)

## Classify each slide into proposal sections

In [ ]:
import json
import re
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def has_table_data(tables_json):
    if tables_json is None:
        return False
    try:
        tables = json.loads(tables_json)
        return len(tables) > 0
    except:
        return False


def contains_dollar_amount(text):
    if text is None:
        return False

    # Matches $, $18,500, ($15,000), $ 1,000.00, etc.
    return bool(re.search(r"\(?\$\s?[\d,]+(?:\.\d{2})?\)?", text))


def detect_scope_language(text):
    if text is None:
        return False

    t = text.lower()

    scope_terms = [
        "scope",
        "scope of work",
        "proposed scope",
        "proposed work",
        "summary",
        "activities",
        "deliverables",
        "workstream"
    ]

    return any(term in t for term in scope_terms)


def classify_slide_type(cleaned_text, table_text, tables_json):
    combined = " ".join([x for x in [cleaned_text, table_text] if x])
    has_tables = has_table_data(tables_json)

    if contains_dollar_amount(combined):
        return "price_slide"

    if detect_scope_language(combined):
        return "scope_slide"

    if has_tables:
        return "structured_content_slide"

    return "narrative"

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
classify_slide_type_udf = udf(classify_slide_type, StringType())

typed_df = cleaned_df.withColumn(
    "slide_type",
    classify_slide_type_udf(
        col("cleaned_text"),
        col("table_text"),
        col("tables_json")
    )
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(
    typed_df
    .groupBy("slide_type")
    .count()
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
display(typed_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

## Chunk the content for search

In [ ]:
proposal_chunks_df = (
    classified_df
    .filter(col("cleaned_text").isNotNull())
    .filter(col("word_count") >= 10)
    .withColumn(
        "chunk_id",
        sha2(concat_ws("|", col("file_id"), col("slide_number"), col("section_type")), 256)
    )
    .select(
        "chunk_id",
        "file_id",
        "customer_name",
        "file_path",
        "slide_number",
        "slide_title",
        "section_type",
        "cleaned_text",
        "word_count",
    )
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
proposal_chunks_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("proposal_chunks")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [23]:
display(proposal_chunks_df)

StatementMeta(, 2b322473-1461-4c1c-a67a-20f5d995cf3e, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc4fc14e-5d33-4272-ad49-d993670babd3)

## Add quality + reuse signals